In [ ]:
from typing import Annotated, List, TypedDict, Optional
import operator
from typing import TypedDict, List, Optional, Union
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

In [20]:
load_dotenv()

True

In [21]:
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.7)

In [37]:
class GrammarState(TypedDict):
    text: str
    is_good: bool
    iteration: int

In [ ]:
def clean_text(state: GrammarState):
    response = llm.invoke(
        f"Improve this text only like grammatical or spelling dont add extra contents:\n{state['text']}"
    )
    return {"text": response.content, "iteration": state["iteration"] + 1}

In [45]:
def evaluate_text(state: GrammarState):
    response = llm.invoke(
        f"Is this text clear and grammatically correct? Answer yes or no and dont give any extra contents:\n{state['text']}"
    )

    if response == "Yes":
        is_good = True
    else:
        is_good = False
    return {"is_good": is_good}

In [46]:
def should_continue(state: GrammarState):
    if state["is_good"] or state["iteration"] >= 3:
        return "end"
    return "continue"

In [47]:
builder = StateGraph(GrammarState)

builder.add_node("clean", clean_text)
builder.add_node("evaluate", evaluate_text)

builder.add_edge(START, "clean")

builder.add_edge("clean", "evaluate")

builder.add_conditional_edges(
    "evaluate", should_continue, {"continue": "clean", "end": END}
)

app = builder.compile()

In [ ]:
result = app.invoke({"text": "The name is Rohan", "iteration": 0, "is_good": False})

print(result)

{'text': 'The name is Rohan\u202fLibing, Mumbai.', 'is_good': False, 'iteration': 3}


In [ ]:
{
    "text": "## Refined\u202fGuide\u202fto Introducing\u202fYour\u202fName\u202fand\u202fLocation  \n\nBelow are ready‑to‑use phrasing options for different communication channels. Pick the one that fits the tone you need, or blend elements to create a line that feels uniquely yours.\n\n| **Context** | **Suggested phrasing** | **Why it works** |\n|-------------|------------------------|-----------------|\n| **Email signature / business card** | *“Rohan\u202fLibing – Mumbai, India”* |\u202fConcise, professional, and instantly tells the reader who you are and where you’re based. |\n| **LinkedIn headline / résumé** | *“Rohan\u202fLibing\u202f·\u202fMarketing Specialist\u202f·\u202fMumbai, India”* |\u202fUses vertical bars (or dots) to separate key identifiers, making the headline scan‑friendly. |\n| **Social‑media bio (casual)** | *“Hey! I’m Rohan\u202fLibing, writing from Mumbai.”* |\u202fFriendly tone, “writing from” hints at a creative or content‑focused role. |\n| **Formal letter or report** | *“The individual referred to herein is Rohan\u202fLibing, residing in Mumbai, India.”* |\u202fFormal diction and a clear subject‑verb structure eliminate any ambiguity. |\n| **Website tagline** | *“Rohan\u202fLibing – Creative Solutions, Mumbai”* |\u202fPairs your name with a value proposition and location, perfect for a personal brand site. |\n| **Conference badge** | *“Rohan\u202fLibing\u202f|\u202fMarketing\u202f|\u202fMumbai”* |\u202fCompact, uses separators to fit limited space while still conveying role and location. |\n| **Pitch deck slide** | *“Rohan\u202fLibing – Founder & CEO, Mumbai‑based”* |\u202fHighlights leadership and geographic base in a single, impactful line. |\n\n### Quick‑Fix Checklist  \n\n1. **Subject‑Verb Clarity** – Start with a noun or pronoun (“Rohan\u202fLibing”, “The individual”) followed by a verb (“is”, “resides”) when the sentence is longer than a tagline.  \n2. **Consistent Punctuation** – Use commas to separate name, title, and location; avoid stray periods after a name.  \n3. **Explicit Location Cue** – Words like **based in**, **from**, **residing in**, or **Mumbai‑based** make it unmistakable that you’re referring to a place, not a company.  \n4. **Tone Matching** – Choose informal verbs (“Hey”, “writing from”) for casual platforms; stick to formal nouns (“individual”, “referred to herein”) for official documents.  \n5. **Brevity** – Aim for 8‑12 words in signatures and taglines; longer contexts (letters, reports) can afford a full sentence.\n\n### How to Customize  \n\n- **Swap the role**: Replace “Marketing Specialist” with your actual title (e.g., “Product Designer”).  \n- **Add a specialty**: “Creative Solutions” → “Data‑Driven Marketing”.  \n- **Adjust geography**: If you work internationally, you might say “Mumbai‑based, serving global clients”.  \n\n#### Example of a blended line  \n> **Rohan\u202fLibing – Marketing Specialist, based in Mumbai, India**  \n\nThis hybrid keeps the professional tone of a résumé headline while explicitly stating your location.\n\nFeel free to mix, match, and tweak any of the suggestions above until you land on a line that feels both authentic and perfectly suited to the audience you’re addressing. Happy introducing!",
    "is_good": False,
    "iteration": 3,
}